In [1]:
import os
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


Using: cuda


In [2]:
PROBLEM = "simple"

def simple_true(x, y, t):
    return np.sin(x) * np.cos(y) * np.exp(-5.0 * t)

def tgv_fields(x, y, t, k=1.0, nu=1.0):
    decay_uv = np.exp(-2.0 * nu * k**2 * t)
    decay_p  = np.exp(-4.0 * nu * k**2 * t)

    u = -np.cos(k * x) * np.sin(k * y) * decay_uv
    v =  np.sin(k * x) * np.cos(k * y) * decay_uv
    p = -0.25 * (np.cos(2.0 * k * x) + np.cos(2.0 * k * y)) * decay_p
    ke = u**2 + v**2
    return u, v, p, ke

def scalar_true(x, y, t, problem=PROBLEM):
    if problem == "simple":
        return simple_true(x, y, t)

    u, v, p, ke = tgv_fields(x, y, t)

    if problem == "tgv_u":
        return u
    elif problem == "tgv_v":
        return v
    elif problem == "tgv_p":
        return p
    elif problem == "tgv_ke":
        return ke
    else:
        raise ValueError("Unknown PROBLEM. Use 'simple', 'tgv_u', 'tgv_v', 'tgv_p', or 'tgv_ke'.")



In [ ]:
class MLP(nn.Module):  
    def __init__(self, input_size, hidden_size, number_outputs, depth, act): 
        super(MLP, self).__init__()  
        self.layers = nn.ModuleList() 
        self.act = act 

       
        self.layers.append(nn.Linear(input_size, hidden_size))
        for i in range(depth - 2):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
        self.layers.append(nn.Linear(hidden_size, number_outputs)) 

    def forward(self, x, final_act=False):
        for j in range(len(self.layers) - 1):
            x = self.act(self.layers[j](x))
        x = self.layers[-1](x)
        if not final_act:
            return x
        else:
            return torch.relu(x)


class DeepONet(nn.Module):
    def __init__(
        self,
        num_sample_points_inputfunc, 
        trunk_input_coordinate_dimension, 
        num_neurons=256,
        num_layers=6,
        branch_output_dimension=256,
        trunk_output_dimension=256,
        number_final_output=1, 
        act_module=nn.Tanh(),
    ):
        super(DeepONet, self).__init__()
        assert branch_output_dimension == trunk_output_dimension 

        self.trunk_output_dimension = trunk_output_dimension
        self.number_final_output = number_final_output

        self.branch_net = MLP(
            input_size=num_sample_points_inputfunc,
            hidden_size=num_neurons,
            number_outputs=branch_output_dimension,
            depth=num_layers,
            act=act_module,
        )

        self.trunk_net = MLP(
            input_size=trunk_input_coordinate_dimension,
            hidden_size=num_neurons,
            number_outputs=trunk_output_dimension,
            depth=num_layers,
            act=act_module,
        )

        self.bias = nn.ParameterList(
            [nn.Parameter(torch.ones(1), requires_grad=True)
             for _ in range(self.number_final_output)]
        ) 

    @staticmethod
    def _to_tensor(x):     
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x)
        return x.to(torch.float32)

    def forward(self, X_branch, X_trunk):
        """
        X_branch: (1,400)
        X_trunk:  (400,3)
        Returns:  (1,400)
        """
        X_branch = self._to_tensor(X_branch)
        X_trunk  = self._to_tensor(X_trunk)

        B = self.branch_net(X_branch)                 
        T = self.trunk_net(X_trunk, final_act=True)   

        output = B @ T.t() + self.bias[0]                
        return output

In [4]:
Nx = Ny = Nt = 20

epochs = 1000
lr = 1e-4
weight_decay = 1e-5
PRINT_EVERY = 100

TRAIN_MODEL = True
LOAD_PRETRAINED_MODEL = False

SAVE_DIR = f"deeponet_outputs_{PROBLEM}"
HEATMAP_DIR = os.path.join(SAVE_DIR, "heatmaps")
MODEL_PATH = os.path.join(SAVE_DIR, f"deeponet_model_{PROBLEM}.pth")
BEST_MODEL_PATH = os.path.join(SAVE_DIR, f"deeponet_best_model_{PROBLEM}.pth")
LOG_PATH = os.path.join(SAVE_DIR, f"logs_{PROBLEM}.npz")
RESULTS_TXT_PATH = os.path.join(SAVE_DIR, f"final_results_{PROBLEM}.txt")
ZIP_PATH = f"deeponet_outputs_{PROBLEM}.zip"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(HEATMAP_DIR, exist_ok=True)
os.makedirs(os.path.join(HEATMAP_DIR, "assimilation"), exist_ok=True)
os.makedirs(os.path.join(HEATMAP_DIR, "forecasting"), exist_ok=True)
os.makedirs(os.path.join(HEATMAP_DIR, "ood"), exist_ok=True)

In [ ]:
if PROBLEM == "simple":
    
    x_train = np.linspace(0.0, 1.0, Nx)
    y_train = np.linspace(0.0, 1.0, Ny)


    dx = 1.0 / (Nx - 1)
    dy = 1.0 / (Ny - 1)
    x_assim = np.linspace(0.0, 1.0, Nx, endpoint=False) + 0.5 * dx
    y_assim = np.linspace(0.0, 1.0, Ny, endpoint=False) + 0.5 * dy


    x_ood = np.linspace(1.0, 2.0, Nx)
    y_ood = np.linspace(1.0, 2.0, Ny)

else:
    x_train = np.linspace(0.0, 2.0 * np.pi, Nx)
    y_train = np.linspace(0.0, 2.0 * np.pi, Ny)

    shift_x = (2.0 * np.pi) / (2 * Nx)
    shift_y = (2.0 * np.pi) / (2 * Ny)
    x_assim = (np.linspace(0.0, 2.0 * np.pi, Nx, endpoint=False) + shift_x) % (2.0 * np.pi)
    y_assim = (np.linspace(0.0, 2.0 * np.pi, Ny, endpoint=False) + shift_y) % (2.0 * np.pi)


    x_ood = np.linspace(0.5, 2.0 * np.pi + 0.5, Nx)
    y_ood = np.linspace(0.5, 2.0 * np.pi + 0.5, Ny)

t_train = np.linspace(0.0, 1.0, Nt)


dt = t_train[1] - t_train[0]
t_forecast = t_train[-1] + dt * np.arange(1, Nt + 1)

In [ ]:
def build_field_tensor(xvals, yvals, tvals, problem=PROBLEM):
    Xg, Yg = np.meshgrid(xvals, yvals, indexing="ij")
    out = np.zeros((len(tvals), len(xvals), len(yvals)), dtype=np.float32)
    for k, tk in enumerate(tvals):
        out[k] = scalar_true(Xg, Yg, tk, problem=problem).astype(np.float32)
    return torch.tensor(out, device=device)

def build_coords(xvals, yvals, t_scalar):
    xx, yy = np.meshgrid(xvals, yvals, indexing="ij")
    coords = np.stack(
        [xx.ravel(),
         yy.ravel(),
         np.full(xx.size, t_scalar)],
        axis=1
    ).astype(np.float32)
    return torch.tensor(coords, dtype=torch.float32, device=device)

def build_one_step_dataset(xvals, yvals, tvals, problem=PROBLEM):
    u_all = build_field_tensor(xvals, yvals, tvals, problem=problem)

    branch = []
    trunk = []
    target = []

    for k in range(len(tvals) - 1):
        branch.append(u_all[k].reshape(-1))
        trunk.append(build_coords(xvals, yvals, tvals[k + 1]))
        target.append(u_all[k + 1].reshape(-1))

    branch = torch.stack(branch).to(device)
    target = torch.stack(target).to(device)
    trunk = [Tk.to(device) for Tk in trunk]

    return u_all, branch, trunk, target


u_train_all, branch_train, trunk_train, target_train = build_one_step_dataset(
    x_train, y_train, t_train, problem=PROBLEM
)


u_assim_all, branch_assim, trunk_assim, target_assim = build_one_step_dataset(
    x_assim, y_assim, t_train, problem=PROBLEM
)


u_forecast_all = build_field_tensor(x_train, y_train, t_forecast, problem=PROBLEM)
u0_forecast = u_forecast_all[0].reshape(-1)

trunk_forecast = []
target_forecast = []
for k in range(Nt - 1):
    trunk_forecast.append(build_coords(x_train, y_train, t_forecast[k + 1]))
    target_forecast.append(u_forecast_all[k + 1].reshape(-1))

trunk_forecast = [Tk.to(device) for Tk in trunk_forecast]
target_forecast = torch.stack(target_forecast).to(device)


u_ood_all, branch_ood, trunk_ood, target_ood = build_one_step_dataset(
    x_ood, y_ood, t_train, problem=PROBLEM
)

In [7]:
model = DeepONet(
    num_sample_points_inputfunc=Nx * Ny,
    trunk_input_coordinate_dimension=3,
    num_neurons=256,
    num_layers=6,
    branch_output_dimension=256,
    trunk_output_dimension=256,
    number_final_output=1,
    act_module=nn.Tanh(),
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

def eval_assimilation():
    model.eval()
    total = 0.0
    with torch.no_grad():
        for k in range(Nt - 1):
            pred = model(branch_assim[k].unsqueeze(0), trunk_assim[k]).squeeze(0)
            total += criterion(pred, target_assim[k]).item()
    return total / (Nt - 1)

def eval_forecasting():
    model.eval()
    total = 0.0
    with torch.no_grad():
        u_hat = u0_forecast.clone()
        for k in range(Nt - 1):
            pred = model(u_hat.unsqueeze(0), trunk_forecast[k]).squeeze(0)
            total += criterion(pred, target_forecast[k]).item()
            u_hat = pred.detach()
    return total / (Nt - 1)

def eval_ood():
    model.eval()
    total = 0.0
    with torch.no_grad():
        for k in range(Nt - 1):
            pred = model(branch_ood[k].unsqueeze(0), trunk_ood[k]).squeeze(0)
            total += criterion(pred, target_ood[k]).item()
    return total / (Nt - 1)


In [ ]:
train_log = []
assim_log = []
forecast_log = []
ood_log = []

best_forecast_mse = float("inf")
best_epoch = 0

if LOAD_PRETRAINED_MODEL and os.path.exists(BEST_MODEL_PATH):
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from: {BEST_MODEL_PATH}")

    if os.path.exists(LOG_PATH):
        logs = np.load(LOG_PATH)
        train_log = list(logs["train_log"])
        assim_log = list(logs["assim_log"])
        forecast_log = list(logs["forecast_log"])
        ood_log = list(logs["ood_log"])
        print(f"Loaded logs from: {LOG_PATH}")

elif TRAIN_MODEL:
    for epoch in range(1, epochs + 1):
        model.train()
        total_train = 0.0

        for k in range(Nt - 1):
            optimizer.zero_grad()
            pred = model(branch_train[k].unsqueeze(0), trunk_train[k]).squeeze(0)
            loss = criterion(pred, target_train[k])
            loss.backward()
            optimizer.step()
            total_train += loss.item()

        mean_train = total_train / (Nt - 1)

        mean_assim = eval_assimilation()
        mean_forecast = eval_forecasting()
        mean_ood = eval_ood()

        train_log.append(mean_train)
        assim_log.append(mean_assim)
        forecast_log.append(mean_forecast)
        ood_log.append(mean_ood)

        if mean_forecast < best_forecast_mse:
            best_forecast_mse = mean_forecast
            best_epoch = epoch
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_mse": mean_train,
                "assim_mse": mean_assim,
                "forecast_mse": mean_forecast,
                "ood_mse": mean_ood,
            }, BEST_MODEL_PATH)

        if epoch % PRINT_EVERY == 0 or epoch == 1:
            print(
                f"[{PROBLEM}] Epoch {epoch:4d} | "
                f"Train {mean_train:.3e} | "
                f"Assim {mean_assim:.3e} | "
                f"Forecast {mean_forecast:.3e} | "
                f"OOD {mean_ood:.3e}"
            )

    torch.save({
        "epoch": epochs,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_log": train_log,
        "assim_log": assim_log,
        "forecast_log": forecast_log,
        "ood_log": ood_log,
    }, MODEL_PATH)

    np.savez(
        LOG_PATH,
        train_log=np.array(train_log),
        assim_log=np.array(assim_log),
        forecast_log=np.array(forecast_log),
        ood_log=np.array(ood_log),
    )

    print(f"\nSaved final model to: {MODEL_PATH}")
    print(f"Saved best model  to: {BEST_MODEL_PATH}")
    print(f"Saved logs        to: {LOG_PATH}")

else:
    raise ValueError("No model available. Set TRAIN_MODEL=True or LOAD_PRETRAINED_MODEL=True.")

if os.path.exists(BEST_MODEL_PATH):
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Using best saved model from epoch {checkpoint.get('epoch', 'unknown')}")


[simple] Epoch    1 | Train 2.850e-01 | Assim 6.297e-01 | Forecast 4.853e+00 | OOD 2.068e-01
[simple] Epoch  100 | Train 2.759e-04 | Assim 3.311e-04 | Forecast 3.013e-04 | OOD 1.154e-02
[simple] Epoch  200 | Train 5.607e-03 | Assim 6.039e-03 | Forecast 3.180e-04 | OOD 5.051e-03


In [ ]:
print(f" Final Results (DeepONet | {PROBLEM}) ")

best_train_mse = float(min(train_log))
best_train_epoch = 1 + int(np.argmin(train_log))
final_train_mse = float(train_log[-1])

best_assim_mse = float(min(assim_log))
best_assim_epoch = 1 + int(np.argmin(assim_log))
final_assim_mse = float(assim_log[-1])

best_forecast_mse = float(min(forecast_log))
best_forecast_epoch = 1 + int(np.argmin(forecast_log))
final_forecast_mse = float(forecast_log[-1])

best_ood_mse = float(min(ood_log))
best_ood_epoch = 1 + int(np.argmin(ood_log))
final_ood_mse = float(ood_log[-1])

print("TRAINING")
print(f"Best epoch      : {best_train_epoch}")
print(f"Best Train MSE  : {best_train_mse:.6e}")
print(f"Final Train MSE : {final_train_mse:.6e}")
print()

print("ASSIMILATION")
print(f"Best epoch      : {best_assim_epoch}")
print(f"Best Test MSE   : {best_assim_mse:.6e}")
print(f"Final Test MSE  : {final_assim_mse:.6e}")
print()

print("FORECASTING")
print(f"Best epoch      : {best_forecast_epoch}")
print(f"Best Test MSE   : {best_forecast_mse:.6e}")
print(f"Final Test MSE  : {final_forecast_mse:.6e}")
print()

print("OOD")
print(f"Best epoch      : {best_ood_epoch}")
print(f"Best Test MSE   : {best_ood_mse:.6e}")
print(f"Final Test MSE  : {final_ood_mse:.6e}")
print()

with open(RESULTS_TXT_PATH, "w") as f:
    f.write(f"Final Results (DeepONet | {PROBLEM})")

    f.write("TRAINING\n")
    f.write(f"Best epoch      : {best_train_epoch}\n")
    f.write(f"Best Train MSE  : {best_train_mse:.6e}\n")
    f.write(f"Final Train MSE : {final_train_mse:.6e}\n\n")

    f.write("ASSIMILATION\n")
    f.write(f"Best epoch      : {best_assim_epoch}\n")
    f.write(f"Best Test MSE   : {best_assim_mse:.6e}\n")
    f.write(f"Final Test MSE  : {final_assim_mse:.6e}\n\n")

    f.write("FORECASTING\n")
    f.write(f"Best epoch      : {best_forecast_epoch}\n")
    f.write(f"Best Test MSE   : {best_forecast_mse:.6e}\n")
    f.write(f"Final Test MSE  : {final_forecast_mse:.6e}\n\n")

    f.write("OOD\n")
    f.write(f"Best epoch      : {best_ood_epoch}\n")
    f.write(f"Best Test MSE   : {best_ood_mse:.6e}\n")
    f.write(f"Final Test MSE  : {final_ood_mse:.6e}\n\n")

print(f"Saved results summary to: {RESULTS_TXT_PATH}")

In [ ]:
epochs_arr = np.arange(1, len(train_log) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_arr, train_log, label="Train MSE", linewidth=2)
plt.plot(epochs_arr, assim_log, label="Assimilation MSE", linewidth=2)
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("MSE (log scale)")
plt.title(f"DeepONet Training Curve ({PROBLEM}: Train vs Assimilation)")
plt.grid(True, which="both", ls="--", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "curve_train_vs_assimilation.png"), dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_arr, train_log, label="Train MSE", linewidth=2)
plt.plot(epochs_arr, forecast_log, label="Forecasting MSE", linewidth=2)
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("MSE (log scale)")
plt.title(f"DeepONet Training Curve ({PROBLEM}: Train vs Forecasting)")
plt.grid(True, which="both", ls="--", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "curve_train_vs_forecasting.png"), dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_arr, train_log, label="Train MSE", linewidth=2)
plt.plot(epochs_arr, ood_log, label="OOD MSE", linewidth=2)
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("MSE (log scale)")
plt.title(f"DeepONet Training Curve ({PROBLEM}: Train vs OOD)")
plt.grid(True, which="both", ls="--", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "curve_train_vs_ood.png"), dpi=300)
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    assim_preds_all = []
    for k in range(Nt - 1):
        pred = model(branch_assim[k].unsqueeze(0), trunk_assim[k]).squeeze(0)
        assim_preds_all.append(pred)
    assim_preds_all = torch.stack(assim_preds_all)

    forecast_preds_all = []
    u_hat = u0_forecast.clone()
    for k in range(Nt - 1):
        pred = model(u_hat.unsqueeze(0), trunk_forecast[k]).squeeze(0)
        forecast_preds_all.append(pred)
        u_hat = pred.detach()
    forecast_preds_all = torch.stack(forecast_preds_all)

    ood_preds_all = []
    for k in range(Nt - 1):
        pred = model(branch_ood[k].unsqueeze(0), trunk_ood[k]).squeeze(0)
        ood_preds_all.append(pred)
    ood_preds_all = torch.stack(ood_preds_all)

assim_vmin = min(target_assim.min().item(), assim_preds_all.min().item())
assim_vmax = max(target_assim.max().item(), assim_preds_all.max().item())

forecast_vmin = min(target_forecast.min().item(), forecast_preds_all.min().item())
forecast_vmax = max(target_forecast.max().item(), forecast_preds_all.max().item())

ood_vmin = min(target_ood.min().item(), ood_preds_all.min().item())
ood_vmax = max(target_ood.max().item(), ood_preds_all.max().item())

assim_err_vmax = torch.abs(assim_preds_all - target_assim).max().item()
forecast_err_vmax = torch.abs(forecast_preds_all - target_forecast).max().item()
ood_err_vmax = torch.abs(ood_preds_all - target_ood).max().item()

def plot_transition_assimilation(k, save=True, show=False):
    pred_img = assim_preds_all[k].detach().cpu().numpy().reshape(Nx, Ny)
    true_img = target_assim[k].detach().cpu().numpy().reshape(Nx, Ny)
    err_img = np.abs(pred_img - true_img)

    vmin = min(true_img.min(), pred_img.min())
    vmax = max(true_img.max(), pred_img.max())
    err_vmax = err_img.max()
    if err_vmax == 0:
        err_vmax = 1e-12

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(true_img.T, origin="lower", vmin=vmin, vmax=vmax)
    plt.title(f"Assim True t={t_train[k+1]:.3f}")
    plt.colorbar()

    plt.subplot(1, 3, 2)
    plt.imshow(pred_img.T, origin="lower", vmin=vmin, vmax=vmax)
    plt.title("Assim Predicted")
    plt.colorbar()

    plt.subplot(1, 3, 3)
    plt.imshow(err_img.T, origin="lower", vmin=0.0, vmax=err_vmax)
    plt.title("Abs Error")
    plt.colorbar()

    plt.suptitle(f"ASSIMILATION | input t={t_train[k]:.3f} -> t={t_train[k+1]:.3f}")
    plt.tight_layout()

    if save:
        plt.savefig(
            os.path.join(HEATMAP_DIR, "assimilation", f"assimilation_step_{k:02d}.png"),
            dpi=300,
            bbox_inches="tight"
        )

    if show:
        plt.show()
    else:
        plt.close()


def plot_transition_forecasting(k, save=True, show=False):
    pred_img = forecast_preds_all[k].detach().cpu().numpy().reshape(Nx, Ny)
    true_img = target_forecast[k].detach().cpu().numpy().reshape(Nx, Ny)
    err_img = np.abs(pred_img - true_img)

    vmin = min(true_img.min(), pred_img.min())
    vmax = max(true_img.max(), pred_img.max())
    err_vmax = err_img.max()
    if err_vmax == 0:
        err_vmax = 1e-12

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(true_img.T, origin="lower", vmin=vmin, vmax=vmax)
    plt.title(f"Forecast True t={t_forecast[k+1]:.3f}")
    plt.colorbar()

    plt.subplot(1, 3, 2)
    plt.imshow(pred_img.T, origin="lower", vmin=vmin, vmax=vmax)
    plt.title("Forecast Predicted (rollout)")
    plt.colorbar()

    plt.subplot(1, 3, 3)
    plt.imshow(err_img.T, origin="lower", vmin=0.0, vmax=err_vmax)
    plt.title("Abs Error")
    plt.colorbar()

    plt.suptitle(
        f"FORECASTING | input t={t_forecast[k]:.3f} -> t={t_forecast[k+1]:.3f} | step k={k}"
    )
    plt.tight_layout()

    if save:
        plt.savefig(
            os.path.join(HEATMAP_DIR, "forecasting", f"forecasting_step_{k:02d}.png"),
            dpi=300,
            bbox_inches="tight"
        )

    if show:
        plt.show()
    else:
        plt.close()


def plot_transition_ood(k, save=True, show=False):
    pred_img = ood_preds_all[k].detach().cpu().numpy().reshape(Nx, Ny)
    true_img = target_ood[k].detach().cpu().numpy().reshape(Nx, Ny)
    err_img = np.abs(pred_img - true_img)

    vmin = min(true_img.min(), pred_img.min())
    vmax = max(true_img.max(), pred_img.max())
    err_vmax = err_img.max()
    if err_vmax == 0:
        err_vmax = 1e-12

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(true_img.T, origin="lower", vmin=vmin, vmax=vmax)
    plt.title(f"OOD True t={t_train[k+1]:.3f}")
    plt.colorbar()

    plt.subplot(1, 3, 2)
    plt.imshow(pred_img.T, origin="lower", vmin=vmin, vmax=vmax)
    plt.title("OOD Predicted")
    plt.colorbar()

    plt.subplot(1, 3, 3)
    plt.imshow(err_img.T, origin="lower", vmin=0.0, vmax=err_vmax)
    plt.title("Abs Error")
    plt.colorbar()

    if PROBLEM == "simple":
        subtitle = f"OOD | input t={t_train[k]:.3f} -> t={t_train[k+1]:.3f} | x,y in [1,2]"
    else:
        subtitle = f"OOD | input t={t_train[k]:.3f} -> t={t_train[k+1]:.3f} | phase-shifted periodic domain"

    plt.suptitle(subtitle)
    plt.tight_layout()

    if save:
        plt.savefig(
            os.path.join(HEATMAP_DIR, "ood", f"ood_step_{k:02d}.png"),
            dpi=300,
            bbox_inches="tight"
        )

    if show:
        plt.show()
    else:
        plt.close()

for k in range(Nt - 1):
    plot_transition_assimilation(k, save=True, show=False)
    plot_transition_forecasting(k, save=True, show=False)
    plot_transition_ood(k, save=True, show=False)

print(f"Saved all heatmaps under: {HEATMAP_DIR}")

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(SAVE_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, start=os.path.dirname(SAVE_DIR))
            zipf.write(full_path, arcname=arcname)

print(f"Created zip archive: {ZIP_PATH}")